# LangChain RAG pipeline (modular)

Hadith RAG over `data/dorar_hadith_full_batch_2.csv`. Each section below is one pipeline stage; change **`CONFIG`** in the next cell to swap models, chunking, or retrieval without touching the rest.

**Run order:** Config → Imports → Load → Split → Embeddings → Vector store → Retriever → LLM → Prompt → Chain → Query.

**Requirements:** `pip install -r requirements.txt` (optional: copy `.env.example` to `.env` for API keys).

In [2]:
# import re
# import pandas as pd
# TASHKEEL = re.compile(r'[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]')
#
# def normalize(text):
#     if pd.isna(text):
#         return text
#     text = re.sub(TASHKEEL, '', text)
#
#     # # normalize Arabic letters
#     # text = re.sub(r'[إأآ]', 'ا', text)
#     # text = re.sub(r'ى', 'ي', text)
#     # text = re.sub(r'ة', 'ه', text)
#     #
#     # # normalize prophet prayer forms
#     # text = re.sub(r'صل[ىي]\s+الله\s+عليه\s+وسلم', 'صلي الله عليه وسلم', text)
#     #
#     # text = re.sub(r'\s+', ' ', text).strip()
#
#     return text
#
# df = pd.read_csv("data/dorar_hadith_full_batch_2.csv")
# df['sharh'] = df['sharh'].apply(normalize)
# df.to_csv("data/dorar_hadith_without_tashkeel.csv")

In [3]:
!python -m pip install -r requirements.txt

  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached instructor-1.15.1-py3-none-any.whl.metadata (12 kB)
  Using cached scikit_network-0.33.5-cp314-cp314-win_amd64.whl
  Using cached docstring_parser-0.18.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached jiter-0.13.0-cp314-cp314-win_amd64.whl.metadata (5.3 kB)
  Using cached rich-14.3.4-py3-none-any.whl.metadata (18 kB)
Using cached ragas-0.4.3-py3-none-any.whl (466 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
Using cached appdirs-1.4.4-py2.py3-none-any.whl (9.6 kB)
Using cached instructor-1.15.1-py3-none-any.whl (178 kB)
Using cached docstring_parser-0.18.0-py3-none-any.whl (22 kB)
Using cached jiter-0.13.0-cp314-cp314-win_amd64.whl (204 kB)
Using cached rich-14.3.4-py3-none-any.whl (310 kB)

  Attempting uninstall: jiter

    Found existing installation: jiter 0.15.

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd

tt = pd.read_csv("data/groupedby_full_hadith_Durar_cleaned_sharh_hadith_only.csv")
tt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14471 entries, 0 to 14470
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   sharh   14471 non-null  object
 1   hadith  14471 non-null  object
dtypes: object(2)
memory usage: 226.2+ KB


In [23]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)
tt[tt['sharh'].str.contains("ووقت أداء صلاة الضحى يبدأ من ارتفاع الشمس")]

,sharh,hadith
5833,"حثنا النبي صلى الله عليه وسلم على التقرب إلى الله بالنوافل والطاعات بعد الفرائض. وفي هذا الحديث تخبر أم هانئ بنت أبي طالب بنت عم النبي صلى الله عليه وسلم: ""أن النبي صلى الله عليه وسلم يوم الفتح"" فتح مكة، وكان في السنة الثامنة من الهجرة، ""ستر عليه""، أي: وضع حوله ستار ليحجب به عورته عن الأعين، ""فاغتسل في الضحى"" فغسل جسده كله بالماء وقت الضحى، وكان ذلك في بيت أم هانئ رضي الله عنها بمكة، ""فصلى ثمان ركعات"" نافلة، ""لا تدري، أقيامها أطول أم سجودها؟""؛ وذلك لأنه كان يتم أركانها من القيام والركوع والسجود. وقيل: إن هذه الصلاة كانت صلاة شكر لله على فتح مكة، إلا أنه اتفق أنها كانت في وقت الضحى. ووقت أداء صلاة الضحى يبدأ من ارتفاع الشمس قيد رمح إلى أن يقوم قائم الظهيرة وقت الزوال، وأقل عدد هو صلاة ركعتين، وقيل: أكثر عدد لها ثمان ركعات لما في حديث أم هانئ، وقيل: إنه لا حد لأكثرها؛ لحديث مسلم عن عائشة رضي الله عنها، قالت: ""كان النبي صلى الله عليه وسلم يصلي الضحى أربعا، ويزيد ما شاء الله"". وإذا صلاها أكثر من ركعتين، فالأفضل له أن يسلم من كل ركعتين؛ لقول النبي صلى الله عليه وسلم: ""صلاة الليل والنهار مثنى مثنى""، وهذا أخرجه أصحاب السنن. وفي الحديث: تعليم للأمة بالتقرب إلى الله بالنوافل والطاعات وقت النعم. وفيه: أن صلاة الضحى سنة عن النبي صلى الله عليه وسلم .",['- أنَّ النَّبيَّ صلَّى اللهُ عليهِ وسلَّمَ يومَ الفتْحِ سُتِرَ عليه، فاغتَسلَ في الضُّحى، فصلَّى ثَمانِ رَكَعاتٍ، لا تَدري، أقِيامُها أَطوَلُ أم سُجودُها؟']
5834,"حثنا النبي صلى الله عليه وسلم على التقرب إلى الله بالنوافل والطاعات بعد الفرائض. وفي هذا الحديث تخبر أم هانئ بنت أبي طالب بنت عم النبي صلى الله عليه وسلم: ""أنها رأت رسول الله صلى الله عليه وسلم يصلي في ثوب واحد، مخالفا بين طرفيه"" وذلك بأن جعل طرفه الأيمن على كتفه الأيسر، وطرفه الأيسر على كتفه الأيمن؛ حتى لا يسقط الثوب وهو يصلي، وذلك أستر للعورة، ""ثمان ركعات بمكة""؛ قيل: كانت نافلة الضحى. وقيل: إن هذه الصلاة كانت صلاة شكر لله على فتح مكة إلا أنه اتفق أنها كانت في وقت الضحى، ""يوم الفتح"" وهو يوم فتح مكة، وكان في السنة الثامنة من الهجرة. ووقت أداء صلاة الضحى يبدأ من ارتفاع الشمس قيد رمح إلى أن يقوم قائم الظهيرة وقت الزوال، وأقل عدد هو صلاة ركعتين، وقيل: أكثر عدد لها ثمان ركعات لما في حديث أم هانئ، وقيل: إنه لا حد لأكثرها؛ لحديث مسلم عن عائشة رضي الله عنها، قالت: ""كان النبي صلى الله عليه وسلم يصلي الضحى أربعا، ويزيد ما شاء الله"". وإذا صلاها أكثر من ركعتين، فالأفضل له أن يسلم من كل ركعتين؛ لقول النبي صلى الله عليه وسلم: ""صلاة الليل والنهار مثنى مثنى""، وهذا أخرجه أصحاب السنن. وفي الحديث: تعليم للأمة بالتقرب إلى الله بالنوافل والطاعات وقت النعم. وفيه: أن صلاة الضحى سنة عن النبي صلى الله عليه وسلم .","['- رأيتُ النبيَّ صلَّى اللهُ عليهِ وسلَّمَ يصلي في ثوبٍ واحدٍ مُلتحِفًا به مخالفًا بين طرَفَيهِ على مَنكِبَيهِ', '- رأيت رسولَ اللهِ صلَّى اللهُ عليهِ وسلمَ يصلي في ثوبٍ واحدٍ مخالفًا بينَ طرَفيه', '- رأَتْ رسولَ اللهِ صلَّى اللهُ عليهِ وسلَّمَ يُصلِّي في ثوبٍ واحدٍ، مُخالِفًا بينَ طرَفَيهِ، ثَمانِ ركَعاتٍ بمكَّةَ، يومَ الفتْحِ.', 'رأَتْ رسولَ اللهِ صلَّى اللهُ عليهِ وسلَّمَ يُصلِّي في ثوبٍ واحدٍ، مُخالِفًا بينَ طرَفَيهِ، ثَمانِ ركَعاتٍ بمكَّةَ، يومَ الفتْحِ.', 'رأَتْ رسولَ اللهِ صلَّى اللهُ عليهِ وسلَّمَ يُصلِّي في ثوبٍ واحدٍ، مُخالِفًا بينَ طرَفَيهِ، ثَمانِ ركَعاتٍ بمكَّةَ، يومَ الفتْحِ.']"


In [1]:
from pathlib import Path

CONFIG = {
  "paths": {
    "data_csv": Path("data/groupedby_full_hadith_Durar_cleaned_sharh_hadith_only.csv"),
    "chroma_dir": Path("chroma_db"),
    "collection_name": "hadith_rag",
  },
  "data": {
    "max_rows": None,          # None = full CSV (large). Start small while experimenting.
    "text_columns": [         # Columns merged into each document body
       "sharh",
    ],
    "metadata_columns":
        ["page_id", "url", "categories", "hadith",
         "rawy", "mohadth",
         "source", "hokm","page","takhrij"],
  },
  "chunking": {
    "chunk_size": 800,
    "chunk_overlap": 120,
    "separators": ["\n\n", "\n", ". ", " ", ""],
  },
  "embeddings": {
    "provider": "huggingface",  # "huggingface" | "openai"
    "model_name": "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2",
    "openai_model": "text-embedding-3-small",
  },
  "vector_store": {
    "persist": True,
    "reset_on_build": True,  # True = delete collection and re-index
  },
  "retriever": {
    "search_type": "similarity",  # "similarity" | "mmr"
    "k": 8,
    "fetch_k": 12,                # used when search_type == "mmr"
    "lambda_mult": 0.5,
  },
  "llm": {
    "provider": "qrok",       # "openai" | "ollama"
    "openai_model": "gpt-4o-mini",
    "ollama_model": "llama3.2",
    "groq_model"  :"llama-3.3-70b-versatile",
    "temperature" : 0.1,
  },
  "prompt": {
    "language": "ar",
    "system_role": (
      "انت عالم دين اسلامي تجاوب علي اسئلة من خلال النص المسند اليك و حاول تجنب تاليف كلام ديني و قم بارفاق الاحاديث الواردة و صحتها و مصدر الاحاديث المتسخدمة"
      "اذا لم تجد جوابا في السياق . ارشده الي استشارة عالم اسلامي افضل للحصول علي اجابة دقيقة"
    ),
  },
}

PROJECT_ROOT = Path(".").resolve()
CONFIG["paths"]["data_csv"] = PROJECT_ROOT / CONFIG["paths"]["data_csv"]
CONFIG["paths"]["chroma_dir"] = PROJECT_ROOT / CONFIG["paths"]["chroma_dir"]

In [2]:
import os
import shutil
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

def cfg(*keys: str) -> Any:
    """Read nested CONFIG values, e.g. cfg('retriever', 'k')."""
    node = CONFIG
    for key in keys:
        node = node[key]
    return node

c:\Users\moham\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [3]:
def row_to_page_content(row: pd.Series) -> str:
    parts = []
    for col in cfg("data", "text_columns"): # for each column
        if col in row.index:
            val = str(row[col]).strip()
            if val and val.lower() != "nan":
                parts.append(f"{col}: {val}")
    return "\n".join(parts)

def load_hadith_documents() -> list[Document]:
    csv_path = cfg("paths", "data_csv")
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    max_rows = cfg("data", "max_rows")
    if max_rows:
        df = df.head(max_rows)

    docs: list[Document] = []
    for _, row in df.iterrows():
        text = row_to_page_content(row)
        if not text.strip():
            continue
        metadata = {
            col: row[col] for col in cfg("data", "metadata_columns") if col in row.index and pd.notna(row[col])
        }
        docs.append(Document(page_content=text, metadata=metadata))

    print(f"Loaded {len(docs)} documents from {csv_path.name} ({len(df)} rows read)")
    return docs


raw_documents = load_hadith_documents()
raw_documents[0].page_content[:400] if raw_documents else "No documents"

Loaded 14471 documents from groupedby_full_hadith_Durar_cleaned_sharh_hadith_only.csv (14471 rows read)


'sharh: "اللهم لك الحمد أنت كسوتنيه"، أي: أنت الذي رزقتني به من غير حول مني ولا قوة، "أسألك من خيره وخير ما صنع له"، أي: أعني على أن أستعمله في طاعتك وعبادتك، ويكون عونا لي فيهما، "وأعوذ بك من شره وشر ما صنع له"، أي: أن أعصي به أو يكون عونا لي في معصيتك. "قال أبو نضرة"، وهو المنذر بن مالك أحد التابعين: "فكان أصحاب النبي صلى الله عليه وسلم إذا لبس أحدهم ثوبا جديدا، قيل له"، أي: دعي له بقوله: "تبلي"،'

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=cfg("chunking", "chunk_size"),
    chunk_overlap=cfg("chunking", "chunk_overlap"),
    separators=cfg("chunking", "separators"),
)

chunks = text_splitter.split_documents(raw_documents)
print(f"Split into {len(chunks)} chunks (avg ~{sum(len(c.page_content) for c in chunks) // max(len(chunks), 1)} chars)")
chunks[0].page_content[:300] if chunks else None

Split into 37815 chunks (avg ~521 chars)


'sharh: "اللهم لك الحمد أنت كسوتنيه"، أي: أنت الذي رزقتني به من غير حول مني ولا قوة، "أسألك من خيره وخير ما صنع له"، أي: أعني على أن أستعمله في طاعتك وعبادتك، ويكون عونا لي فيهما، "وأعوذ بك من شره وشر ما صنع له"، أي: أن أعصي به أو يكون عونا لي في معصيتك. "قال أبو نضرة"، وهو المنذر بن مالك أحد التابعي'

In [5]:
# !pip uninstall torch torchvision torchaudio

In [6]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [7]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

2.11.0+cu130
13.0
True
1


In [8]:
def build_embeddings():
    provider = cfg("embeddings", "provider")
    if provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model=cfg("embeddings", "openai_model"))
    if provider == "huggingface":
        from langchain_community.embeddings import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(model_name=cfg("embeddings", "model_name"), 
            model_kwargs={"device": "cuda"},
            encode_kwargs={
                "batch_size": 256
            })
    raise ValueError(f"Unknown embeddings provider: {provider}")


embeddings = build_embeddings()
# Quick sanity check (optional; comment out on slow machines)
# len(embeddings.embed_query("اختبار"))
print(f"Embeddings ready: {cfg('embeddings', 'provider')} / {cfg('embeddings', 'model_name') if cfg('embeddings', 'provider') == 'huggingface' else cfg('embeddings', 'openai_model')}")

C:\Users\moham\AppData\Local\Temp\ipykernel_15960\1321039682.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
C:\Users\moham\AppData\Local\Temp\ipykernel_15960\1321039682.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name=cfg("embeddings", "model_name"),


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings ready: huggingface / Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2


In [9]:
from langchain_chroma import Chroma

chroma_dir = cfg("paths", "chroma_dir")
collection = cfg("paths", "collection_name")

if cfg("vector_store", "reset_on_build") and chroma_dir.exists():
    shutil.rmtree(chroma_dir)
    print(f"Removed {chroma_dir}")

vectorstore = Chroma(
    collection_name=collection,
    embedding_function=embeddings,
    persist_directory=str(chroma_dir) if cfg("vector_store", "persist") else None,
)

def _collection_has_vectors(vs: Chroma) -> bool:
    data = vs.get(limit=1)
    return bool(data.get("ids"))


# Index only if collection is empty (re-run safe)
if not _collection_has_vectors(vectorstore):
    # vectorstore.add_documents(chunks)
    # print(f"Indexed {len(chunks)} chunks into '{collection}'")

    from tqdm import tqdm
    batch_size = 512

    for i in tqdm(range(0, len(chunks), batch_size)):
        vectorstore.add_documents(
            chunks[i:i+batch_size]
        )
    print(f"Indexed {len(chunks)} chunks into '{collection}'")

else:
    n = len(vectorstore.get().get("ids", []))
    print(f"Using existing index: {n} vectors in '{collection}'")

vectorstore

100%|██████████| 74/74 [11:58<00:00,  9.71s/it]

Indexed 37815 chunks into 'hadith_rag'


In [10]:
# 3 -> 2135
# x -> 30,000
def build_retriever():
    search_type = cfg("retriever", "search_type")
    k = cfg("retriever", "k")

    if search_type == "similarity":
        return vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})
    if search_type == "mmr":
        return vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": k,
                "fetch_k": cfg("retriever", "fetch_k"),
                "lambda_mult": cfg("retriever", "lambda_mult"),
            },
        )
    raise ValueError(f"Unknown search_type: {search_type}")


retriever = build_retriever()
print(f"Retriever: {cfg('retriever', 'search_type')}, k={cfg('retriever', 'k')}")

Retriever: similarity, k=8


In [11]:
def build_llm():
    provider = cfg("llm", "provider")
    temperature = cfg("llm", "temperature")

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=cfg("llm", "openai_model"), temperature=temperature)
    if provider == "ollama":
        from langchain_community.chat_models import ChatOllama
        return ChatOllama(model=cfg("llm", "ollama_model"), temperature=temperature)
    if provider == "qrok":
        from langchain_groq import ChatGroq
        return ChatGroq(model=cfg("llm", "groq_model"), temperature=temperature)
        # max_tokens=1024
    raise ValueError(f"Unknown llm provider: {provider}")

llm = build_llm()
print(f"LLM: {cfg('llm', 'provider')}")

LLM: qrok


In [12]:
def format_docs(docs: list[Document]) -> str:
    blocks = []
    for i, doc in enumerate(docs, 1):
        meta = ", ".join(f"{k}={v}" for k, v in doc.metadata.items())
        blocks.append(f"[{i}] ({meta})\n{doc.page_content}")
    return "\n\n---\n\n".join(blocks)


RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", cfg("prompt", "system_role")),
    (
        "human",
        "السياق المسترجع:\n{context}\n\n"
        "السؤال: {question}\n\n"
        f"أجب باللغة: {cfg('prompt', 'language')}. اذكر page_id عند الاقتباس إن وُجد.",
    ),
])

prompt = RAG_PROMPT
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='انت عالم دين اسلامي تجاوب علي اسئلة من خلال النص المسند اليك و حاول تجنب تاليف كلام ديني و قم بارفاق الاحاديث الواردة و صحتها و مصدر الاحاديث المتسخدمةاذا لم تجد جوابا في السياق . ارشده الي استشارة عالم اسلامي افضل للحصول علي اجابة دقيقة'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='السياق المسترجع:\n{context}\n\nالسؤال: {question}\n\nأجب باللغة: ar. اذكر page_id عند الاقتباس إن وُجد.'), additional_kwargs={})])

In [14]:
# LCEL chain: question -> retrieve -> format -> prompt -> llm -> text
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Optional: inspect retrieval only (no LLM cost)
def retrieve(question: str, k: int | None = None):
    docs = retriever.invoke(question)
    if k:
        docs = docs[:k]
    return docs

rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000018AE60FD940>, search_kwargs={'k': 8})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='انت عالم دين اسلامي تجاوب علي اسئلة من خلال النص المسند اليك و حاول تجنب تاليف كلام ديني و قم بارفاق الاحاديث الواردة و صحتها و مصدر الاحاديث المتسخدمةاذا لم تجد جوابا في السياق . ارشده الي استشارة عالم اسلامي افضل للحصول علي اجابة دقيقة'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='السياق المسترجع:\n{context}\n\nالسؤال: {question}\n\nأجب باللغة: ar. اذكر page_id عند الاقتباس إن وُجد.'), addi

In [15]:
def ask(question: str, *, show_sources: bool = True) -> str:
    if show_sources:
        print("--- Retrieved chunks ---")
        for doc in retrieve(question):
            print(f"page_id={doc.metadata.get('page_id')} | {doc.page_content}... | {doc.metadata}")
        print("--- Answer ---")
    return rag_chain.invoke(question)

QUESTION = "ما حكم سجود السهو إذا زاد الإمام في الصلاة؟"
answer = ask(QUESTION)
print(answer)

--- Retrieved chunks ---
page_id=None | sharh: الصلاة عماد الدين، وعلى العبد أن يلزم فيها الخشوع والتدبر، وعدم الانشغال بأحوال الدنيا، ولكنه قد يسهو فيها، فينقص أو يزيد في بعض أفعالها، وهذا السهو يحتاج إلى ما يجبره، وقد شرع سجود السهو لمثل ذلك.وفي هذا الحديث يروي عبد الله بن مسعود رضي الله عنه أن النبي صلى الله عليه وسلم سها يوما في صلاة الظهر فصلاها خمس ركعات، فلما سلم من صلاته قال له بعض من صلى معه: أزيد في الصلاة؟ ومعناه السؤال عن حدوث شيء من الوحي يوجب تغيير حكم الصلاة بالزيادة على ما كانت معهودة، فقال لهم النبي صلى الله عليه وسلم: «وما ذاك؟»، وهو سؤال من لم يشعر بما وقع منه، فقالوا: صليت خمس ركعات، فثنى رجليه، أي: فجلس على هيئة القعود للتشهد، ثم سجد سجدتين للسهو، والمقصود من الحديث: أن من سها في صلاته وزاد فيها وسلم وهو ناس، ثم ذكر بعد سلامه، فإنه يستقبل القبلة ويسجد للسهو؛ فإن سجود السهو من تمام الصلاة، ولو كان بعد السلام،... | {'hadith': "['- صَلَّى النَّبيُّ صلَّى اللهُ عليه وسلَّم الظُّهرَ خَمسًا، فقالوا: أزِيدَ في الصَّلاةِ؟ قال: وما ذاكَ؟ قالوا: صَلَّيتَ خَمسًا، فثَنى رِجلَي

## Customization cheat sheet

| Goal | Change in `CONFIG` |
|------|-------------------|
| Use full dataset | `"max_rows": None` |
| Rebuild vector DB | `"reset_on_build": True` (run vector-store cell once) |
| More context per answer | Increase `retriever.k` or `chunk_size` |
| Diverse retrieval | `"search_type": "mmr"` |
| Local LLM | `"llm": {"provider": "ollama", ...}` + run Ollama |
| OpenAI embeddings | `"embeddings": {"provider": "openai", ...}` |
| Different fields in chunks | Edit `data.text_columns` / `metadata_columns` |
| Swap only the prompt | Edit `prompt.system_role` or the `RAG_PROMPT` cell |

**Swap a component:** re-run from that cell downward (e.g. new embeddings → vector store → retriever → chain).

In [42]:
import pandas as pd
eval = pd.read_csv("data/HAQA.csv")

In [43]:
eval.head(1)

,Record_Id,Question_Id,Question_Text,Quetion_Type,Question_Start_Word,Answer_ID,Full_Answer,Expert_Commentary,Hadith_Full_Answer,Hadith_Matn,Answer-Instances,Source_Name,Source_Link,Credibility,Question_ID_in_the_Orignal_Dataset
0,1,1,كيف نعبد الله؟,D,كيف,1.0,كَمَا أمرنا الله ورسوله مَعَ الإخلاص ( وَمَا أُمروا إِلاَّ ليَعبدوا اللهَ مخلصينَ لَهُ الدّين) [البيّنة: 5]. ( مَن عمِلَ عمَلاً لَيْسَ عَلَيهِ أمْرُنا فَهُوَ رَدٌّ) [أَيْ مردود] (رواه مسلم),كَمَا أمرنا الله ورسوله مَعَ الإخلاص,( مَن عمِلَ عمَلاً لَيْسَ عَلَيهِ أمْرُنا فَهُوَ رَدٌّ) [أَيْ مردود] (رواه مسلم),مَن عمِلَ عمَلاً لَيْسَ عَلَيهِ أمْرُنا فَهُوَ رَدٌّ,مَن عمِلَ عمَلاً لَيْسَ عَلَيهِ أمْرُنا فَهُوَ رَدٌّ,كتاب عقيدة كل مسلم في سؤال و جواب لمحمد بن جميل زينو,https://www.noor-book.com/%D9%83%D8%AA%D8%A7%D8%A8-%D8%B9%D9%82%D9%8A%D8%AF%D8%A9-%D9%83%D9%84-%D9%85%D8%B3%D9%84%D9%85-%D9%81%D9%8A-%D8%B3%D8%A4%D8%A7%D9%84-%D9%88-%D8%AC%D9%88%D8%A7%D8%A8-pdf,yes,2


In [44]:
eval.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1597 entries, 0 to 1596
Data columns (total 15 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Record_Id                           1597 non-null   int64  
 1   Question_Id                         1597 non-null   int64  
 2   Question_Text                       1597 non-null   object 
 3   Quetion_Type                        1597 non-null   object 
 4   Question_Start_Word                 1597 non-null   object 
 5   Answer_ID                           1597 non-null   float64
 6   Full_Answer                         1597 non-null   object 
 7   Expert_Commentary                   1579 non-null   object 
 8   Hadith_Full_Answer                  1597 non-null   object 
 9   Hadith_Matn                         1597 non-null   object 
 10  Answer-Instances                    1597 non-null   object 
 11  Source_Name                         1597 no

In [20]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)
eval[['Question_Text', 'Hadith_Matn']].head(10)

,Question_Text,Hadith_Matn
0,كيف نعبد الله؟,مَن عمِلَ عمَلاً لَيْسَ عَلَيهِ أمْرُنا فَهُوَ رَدٌّ
1,هل نعبد الله خوفا وطمعا؟,أسألُ اللهَ الجنّة وأعوذ بِهِ مِن النّار
2,ما هو الإحسان في العبادة؟,الإحسانُ أَنْ تعبُدَ اللهَ كأنّك تراه فإن لَمْ تكن تراهُ فإنَّه يراك
3,ما معنى لا إله إلا الله؟,من قَالَ لآ إله إِلاَّ الله وكَفَرَ بِمَا يُعبدُ مِن دون الله حَرُمَ مالُه ودمُه
4,ما هو التوحيد في صفات الله؟,ينزِلُ ربُّنا تبارك وتعالى فِي كلّ ليلةٍ إِلَى السّمَاء الدُّنْيَا
5,ما هي فائدة التوحيد للمسلم؟,حَقُّ العباد عَلَى الله أَنْ لاَ يُعَذّب من لاَ يُشرك بِهِ شيئاً
6,أين الله؟,إنّ الله كتب كتاباً إنّ رحمتي سبقت غضبي فَهُوَ مكتوبٌ عنده فَوْقَ العرش
7,هل الله معنا بذاته أم بعلمه؟,إنّكم تدعون سميعاً قريباً وَهُوَ معكم
8,ما هو أعظم الذنوب؟,سُئِلَ صلى الله عليه وسلم أيُّ الذَّنب أعظم؟ قَالَ: أَنْ تدعو للهِ ندّاً وَهُوَ خلقك
9,ما هو الشرك الأكبر؟,أكبرُ الكبائر الإشراكُ باللهِ


In [22]:
eval[eval['Question_Id'] == 445]

,Record_Id,Question_Id,Question_Text,Quetion_Type,Question_Start_Word,Answer_ID,Full_Answer,Expert_Commentary,Hadith_Full_Answer,Hadith_Matn,Answer-Instances,Source_Name,Source_Link,Credibility,Question_ID_in_the_Orignal_Dataset
512,513,445,الاستجمار بكم يكون من الحجارة ؟,D,بكم,1.0,إذا استجمر فليوتر، ولا يكون بأقل من ثلاث يعني: بثلاث، أو خمس، أو سبع، وهكذا. ⇦ والدليل حديث أَبِي هُرَيْرَةَ رضي الله عنه، أَنَّ رَسُولَ الله ﷺ قَالَ: «مَنِ اسْتَجْمَرَ فَليُوتِرْ...». الحديث. رواه البخاري (161)، ومسلم (237). ⇦ وَعَنْ سَلمَانَ رضي الله عنه، قَالَ: «نَهَانَا رَسُولُ الله ﷺ أَنْ نَسْتَنْجِيَ بِأَقَلَّ مِنْ ثَلَاثَةِ أَحْجَارٍ». رواه مسلم (262).,إذا استجمر فليوتر، ولا يكون بأقل من ثلاث يعني: بثلاث، أو خمس، أو سبع، وهكذا.,حديث أَبِي هُرَيْرَةَ رضي الله عنه، أَنَّ رَسُولَ الله ﷺ قَالَ: «مَنِ اسْتَجْمَرَ فَليُوتِرْ...». الحديث. رواه البخاري (161)، ومسلم (237).,أَنَّ رَسُولَ الله ﷺ قَالَ: مَنِ اسْتَجْمَرَ فَليُوتِرْ,فَليُوتِرْ,الاستدلال على كنز الأطفال للدكتور فيصل بن مسفر بن معوض الزنامي الوادعي,https://alilmia.com/sub_book91_5410.html,yes,٧٦٥
513,514,445,الاستجمار بكم يكون من الحجارة ؟,D,بكم,2.0,إذا استجمر فليوتر، ولا يكون بأقل من ثلاث يعني: بثلاث، أو خمس، أو سبع، وهكذا. ⇦ والدليل حديث أَبِي هُرَيْرَةَ رضي الله عنه، أَنَّ رَسُولَ الله ﷺ قَالَ: «مَنِ اسْتَجْمَرَ فَليُوتِرْ...». الحديث. رواه البخاري (161)، ومسلم (237). ⇦ وَعَنْ سَلمَانَ رضي الله عنه، قَالَ: «نَهَانَا رَسُولُ الله ﷺ أَنْ نَسْتَنْجِيَ بِأَقَلَّ مِنْ ثَلَاثَةِ أَحْجَارٍ». رواه مسلم (262).,إذا استجمر فليوتر، ولا يكون بأقل من ثلاث يعني: بثلاث، أو خمس، أو سبع، وهكذا.,عَنْ سَلمَانَ رضي الله عنه، قَالَ: «نَهَانَا رَسُولُ الله ﷺ أَنْ نَسْتَنْجِيَ بِأَقَلَّ مِنْ ثَلَاثَةِ أَحْجَارٍ». رواه مسلم (262).,قَالَ: «نَهَانَا رَسُولُ الله ﷺ أَنْ نَسْتَنْجِيَ بِأَقَلَّ مِنْ ثَلَاثَةِ أَحْجَارٍ»,نَهَانَا رَسُولُ الله ﷺ أَنْ نَسْتَنْجِيَ بِأَقَلَّ مِنْ ثَلَاثَةِ أَحْجَارٍ,الاستدلال على كنز الأطفال للدكتور فيصل بن مسفر بن معوض الزنامي الوادعي,https://alilmia.com/sub_book91_5410.html,yes,٧٦٥


---
## Evaluation (RAG trials)

Compare pipeline settings using:

1. **Hadith-in-retrieval (HAQA)** — for each `Question_Text`, did `Hadith_Matn` appear in retrieved chunks **before** the LLM? (checks `page_content` + `hadith_1` metadata)
2. **RAGAS** — faithfulness, answer relevancy, context precision/recall (needs LLM + embeddings; uses `Expert_Commentary` as reference when available)

**Prerequisites:** run all pipeline cells above (`retriever`, `rag_chain`, `llm`, `embeddings`).

Results are saved under `eval_runs/` so you can diff runs after changing `CONFIG` / `EVAL_CONFIG`.

In [ ]:
# --- eval_config.py (notebook trial; move to module later) ---
from datetime import datetime, timezone
from pathlib import Path

EVAL_CONFIG = {
    "haqa_csv": PROJECT_ROOT / "data" / "HAQA.csv",
    "max_samples": None,              # None = full HAQA (slow / costly for RAGAS)
    "random_seed": 42,
    "output_dir": PROJECT_ROOT / "eval_runs",
    "run_name": None,                 # None -> timestamped folder
    # Hadith-in-retrieval
    "retrieval": {
        "enabled": True,
        "min_substring_len": 12,    # ignore very short Hadith_Matn for substring test
        "min_token_overlap": 0.45,  # fallback: |intersection| / |hadith_tokens|
        "check_metadata_hadith_1": True,
    },
    # RAGAS
    "ragas": {
        "enabled": True,
        "metrics": [
            "faithfulness",
            "answer_relevancy",
            "context_precision",
            "context_recall",
        ],
        # TODO : change ground truth to be hadith_matn
        "ground_truth_column": "Expert_Commentary",  # fallback: Full_Answer
        "max_contexts_for_ragas": None,              # None = all retrieved (k from CONFIG)
    },
}

def _eval_run_dir() -> Path:
    name = EVAL_CONFIG["run_name"] or datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    run_dir = EVAL_CONFIG["output_dir"] / name
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

EVAL_RUN_DIR = _eval_run_dir()
print(f"Eval outputs -> {EVAL_RUN_DIR}")

Eval outputs -> D:\ITI_Ai_Intake46\Graduation Project\RAG_pipeline\eval_runs\Grouped sharh v1


In [100]:
koks = retrieve("ما حكم تربيةالدقن")

In [101]:
for hadith in ast.literal_eval(koks[0].metadata['hadith'])[::-1]:
    print(hadith)

أنَّ ابنَ عباسٍ سُئِلَ عمَّن ذَبَحَ دجاجةً فطيَّرَ رأسَها فقال ذكاةٌ وحيَّةٌ
أنَّ ابنَ عباسٍ سُئِلَ عمَّن ذَبَحَ دجاجةً فطيَّرَ رأسَها فقال ذكاةٌ وحيَّةٌ
- أنَّ ابنَ عبَّاسٍ سُئلَ عمَّن ذبحَ دجاجةً فطَيَّرَ رأسَها فقالَ ذَكاةٌ وَحِيَّةٌ
- عن ابنِ عبَّاسٍ سُئِلَ عن ذَبحِ دَجاجَةٍ طُيِّرَ رأسُها، فقال: ذَكاةٌ وَحِيَّةٌ.
- أنَّ ابنَ عباسٍ سُئِلَ عمَّن ذَبَحَ دجاجةً فطيَّرَ رأسَها فقال ذكاةٌ وحيَّةٌ


In [89]:
len(ast.literal_eval(koks[5].metadata['hadith']))

1

In [102]:
all_hadiths = []
import ast
for doc in koks:
    print(ast.literal_eval(doc.metadata['hadith']))
all_hadiths

['- أنَّ ابنَ عباسٍ سُئِلَ عمَّن ذَبَحَ دجاجةً فطيَّرَ رأسَها فقال ذكاةٌ وحيَّةٌ', '- عن ابنِ عبَّاسٍ سُئِلَ عن ذَبحِ دَجاجَةٍ طُيِّرَ رأسُها، فقال: ذَكاةٌ وَحِيَّةٌ.', '- أنَّ ابنَ عبَّاسٍ سُئلَ عمَّن ذبحَ دجاجةً فطَيَّرَ رأسَها فقالَ ذَكاةٌ وَحِيَّةٌ', 'أنَّ ابنَ عباسٍ سُئِلَ عمَّن ذَبَحَ دجاجةً فطيَّرَ رأسَها فقال ذكاةٌ وحيَّةٌ', 'أنَّ ابنَ عباسٍ سُئِلَ عمَّن ذَبَحَ دجاجةً فطيَّرَ رأسَها فقال ذكاةٌ وحيَّةٌ']
['- أتَيْتُ رسولَ اللهِ صلَّى اللهُ عليه وسلَّم بعبدِ اللهِ بنِ أبي طلحةَ ليُحنِّكَه فوافَيْتُه بيدِه المِيسَمُ يسِمُ إبلَ الصَّدقةِ', '- حينَ ولدت أمِّي انطلقتُ بالصَّبيِّ إلى النَّبيِّ صلَّى اللَّهُ عليهِ وسلَّمَ ليحنِّكَهُ، فإذا النَّبيُّ صلَّى اللَّهُ عليهِ وسلَّمَ في مِربَدٍ لَنا، يسِمُ غنمًا قالَ شُعبةُ: أَكْثرُ علمي أنَّهُ قالَ: في آذانِها', '- أنه يعني النبيَّ صلَّى اللهُ عليهِ وسلَّمَ كان يَسِمُ إبلَ الصَّدَقَةِ.', '- غَدَوتُ إلى رَسولِ اللهِ صلَّى اللهُ عليه وسلَّم بعَبدِ اللهِ بنِ أبي طَلحةَ ليُحَنِّكَه، فوافَيتُه في يَدِه الميسَمُ يَسِمُ إبِلَ الصَّدَقةِ.', '- أتَيْت

[]

In [107]:
# --- eval_text_utils.py ---
import re
from dataclasses import dataclass

ARABIC_TASHKEEL = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]")
NON_WORD = re.compile(r"[^\w\s\u0600-\u06FF]+", re.UNICODE)


def normalize_arabic(text: str) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    text = str(text)
    text = ARABIC_TASHKEEL.sub("", text)
    text = text.replace("\ufeff", "")
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = NON_WORD.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def token_set(text: str) -> set[str]:
    return {t for t in normalize_arabic(text).split() if len(t) > 1}


def token_overlap_ratio(a: str, b: str) -> float:
    ta, tb = token_set(a), token_set(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta)

def sequence_matcher(a: str, b: str) -> float:
    from difflib import SequenceMatcher
    return  SequenceMatcher(None, normalize_arabic(a), normalize_arabic(b)).ratio()

# def doc_retrieval_blob(doc: Document) -> str:
#     # TODO : include hadith only
#     # parts = [doc.page_content]
#     parts = []
#     if EVAL_CONFIG["retrieval"]["check_metadata_hadith_1"]:
#         hadiths = doc.metadata.get("hadith")
#         for h in hadiths:
#             if h is not None and str(h).strip() != "nan":
#                 parts.append(str(h))
#     return "\n".join(parts)

In [108]:
# --- eval_retrieval_hadith.py ---
@dataclass
class HadithHitResult:
    hit: bool
    method: str | None          # substring | token_overlap | none
    best_overlap: float
    matched_doc_index: str | None


def hadith_in_retrieved_docs(
    hadith_matn: str,
    all_hadith: list[str], * , 
    min_substring_len: int | None = None, min_token_overlap: float | None = None) -> HadithHitResult:


    min_substring_len = min_substring_len or EVAL_CONFIG["retrieval"]["min_substring_len"]
    min_token_overlap = min_token_overlap or EVAL_CONFIG["retrieval"]["min_token_overlap"]

    target = normalize_arabic(hadith_matn)
    if len(target) < 3:
        return HadithHitResult(False, "none", 0.0, None)

    best_overlap = 0.0
    best_hadith = ""
    for i, hadith in enumerate(all_hadith):
        blob_norm = normalize_arabic(hadith)
        if len(target) >= min_substring_len and target in blob_norm:
            return HadithHitResult(True, "substring", 1.0, blob_norm)
        # overlap = token_overlap_ratio(hadith_matn, doc_retrieval_blob(doc))
        overlap = sequence_matcher(target, blob_norm)
        if overlap > best_overlap:
            best_overlap, best_hadith = overlap, blob_norm

    if best_overlap >= min_token_overlap:
        return HadithHitResult(True, "token_overlap", best_overlap, best_hadith)
    return HadithHitResult(False, "none", best_overlap, best_hadith)


def load_haqa_eval_frame() -> pd.DataFrame:
    path = EVAL_CONFIG["haqa_csv"]
    df = pd.read_csv(path)
    df = df.dropna(subset=["Question_Text", "Hadith_Matn"])
    max_n = EVAL_CONFIG["max_samples"]
    if max_n:
        df = df.sample(n=min(max_n, len(df)), random_state=EVAL_CONFIG["random_seed"])
    return df.reset_index(drop=True)

In [109]:
# --- run retrieval-only eval (no LLM) ---
def run_haqa_retrieval_eval(haqa_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(haqa_df.iterrows()):
        question = row["Question_Text"]
        hadith = row["Hadith_Matn"]
        docs = retrieve(normalize_arabic(question))
        all_hadiths = []
        import ast
        for doc in docs:
            all_hadiths += ast.literal_eval(doc.metadata['hadith'])
        hit = hadith_in_retrieved_docs(hadith, all_hadiths)
        rows.append({
            "Record_Id": row.get("Record_Id"),
            "Question_Id": row.get("Question_Id"),
            "Question_Text": question,
            "Hadith_Matn": hadith,
            "hadith_hit": hit.hit,
            "hit_method": hit.method,
            "best_token_overlap": round(hit.best_overlap, 4),
            # "matched_rank": (hit.matched_doc_index + 1) if hit.matched_doc_index is not None else None,
            # "retrieved_page_ids": [d.metadata.get("page_id") for d in docs],
            "n_retrieved": len(docs),
            "hadith":hit.matched_doc_index
        })
    return pd.DataFrame(rows)


if EVAL_CONFIG["retrieval"]["enabled"]:
    haqa_eval_df = load_haqa_eval_frame()
    retrieval_results_df = run_haqa_retrieval_eval(haqa_eval_df)

    hit_rate = retrieval_results_df["hadith_hit"].mean()
    print(f"Hadith-in-retrieval hit rate: {hit_rate:.1%} ({retrieval_results_df['hadith_hit'].sum()}/{len(retrieval_results_df)})")
    print(retrieval_results_df["hit_method"].value_counts(dropna=False))

    out_path = EVAL_RUN_DIR / "haqa_retrieval_hits.csv"
    retrieval_results_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    import json
    summary_path = EVAL_RUN_DIR / "run_summary.json"
    summary_path.write_text(
        json.dumps(
            {
                "run_dir": str(EVAL_RUN_DIR),
                "n_samples": len(retrieval_results_df),
                "config_snapshot": CONFIG,
                "eval_config": {k: v for k, v in EVAL_CONFIG.items() if k != "haqa_csv"},
                "hadith_hit_rate": float(hit_rate),
                "hit_method_counts": retrieval_results_df["hit_method"].value_counts().to_dict(),
            },
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    print(f"Saved -> {out_path}, {summary_path}")
    retrieval_results_df.head()
else:
    print("Retrieval eval disabled in EVAL_CONFIG")

1597it [04:05,  6.50it/s]

Hadith-in-retrieval hit rate: 56.0% (895/1597)
hit_method
token_overlap    884
none             702
substring         11
Name: count, dtype: int64
Saved -> D:\ITI_Ai_Intake46\Graduation Project\RAG_pipeline\eval_runs\Grouped sharh v1\haqa_retrieval_hits.csv, D:\ITI_Ai_Intake46\Graduation Project\RAG_pipeline\eval_runs\Grouped sharh v1\run_summary.json


In [ ]:
# from difflib import SequenceMatcher

# s1 = "   "
# s2 = "قال رسولُ اللهِ صلَّى اللهُ عليه وسلَّمَ: قُلِ اللَّهُمَّ إنِّي أسأَلُكَ الهُدى والسَّدادَ، واذكُرْ بالهُدى هِدايتَكَ الطَّريقَ، واذكُرْ بالسَّدادِ تَسديدَكَ السَّهمَ، قال: ونَهى -أو نَهاني- عنِ القَسِّيِّ، والمِيثَرةِ، وعنِ الخاتَمِ في السَّبَّابةِ أوِ الوُسْطى."


# score = SequenceMatcher(None, normalize_arabic(s1), normalize_arabic(s2)).ratio()
# print(score)

0.0


## RAGAS

In [110]:
!python -m pip install ragas


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# --- eval_ragas.py ---
import json

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)

RAGAS_METRIC_MAP = {
    "faithfulness": faithfulness,
    "answer_relevancy": answer_relevancy,
    "context_precision": context_precision,
    "context_recall": context_recall,
}
# TODO : change ground truth table

def _ground_truth_from_row(row: pd.Series) -> str:
    col = EVAL_CONFIG["ragas"]["ground_truth_column"]
    if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
        return str(row[col])
    if pd.notna(row.get("Full_Answer")):
        return str(row["Full_Answer"])
    return str(row.get("Expert_Commentary", ""))


def build_ragas_dataset(haqa_df: pd.DataFrame) -> Dataset:
    rows = []
    max_ctx = EVAL_CONFIG["ragas"]["max_contexts_for_ragas"]
    for _, row in haqa_df.iterrows():
        question = row["Question_Text"]
        docs = retrieve(question)
        contexts = [d.page_content for d in docs]
        if max_ctx:
            contexts = contexts[:max_ctx]
        answer = rag_chain.invoke(question)
        rows.append({
            "question": question,
            "answer": answer,
            "contexts": contexts,
            "ground_truth": _ground_truth_from_row(row),
            "hadith_matn": row["Hadith_Matn"],
        })
    return Dataset.from_list(rows)


def run_ragas_eval(haqa_df: pd.DataFrame):
    metric_names = EVAL_CONFIG["ragas"]["metrics"]
    metrics = [RAGAS_METRIC_MAP[m] for m in metric_names]

    # Uses pipeline `llm` + `embeddings` (Groq / HuggingFace, etc.)
    dataset = build_ragas_dataset(haqa_df)
    result = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=llm,
        embeddings=embeddings,
    )
    return result


if EVAL_CONFIG["ragas"]["enabled"]:
    if "haqa_eval_df" not in globals():
        haqa_eval_df = load_haqa_eval_frame()

    ragas_result = run_ragas_eval(haqa_eval_df)
    ragas_per_question_df = ragas_result.to_pandas()

    per_row_path = EVAL_RUN_DIR / "ragas_per_question.csv"
    ragas_per_question_df.to_csv(per_row_path, index=False, encoding="utf-8-sig")

    metric_cols = [c for c in ragas_per_question_df.columns if c in EVAL_CONFIG["ragas"]["metrics"]]
    ragas_mean = ragas_per_question_df[metric_cols].mean(numeric_only=True).to_dict()

    summary = {
        "run_dir": str(EVAL_RUN_DIR),
        "n_samples": len(haqa_eval_df),
        "config_snapshot": CONFIG,
        "eval_config": {k: v for k, v in EVAL_CONFIG.items() if k != "haqa_csv"},
        "ragas_mean": ragas_mean,
        "hadith_hit_rate": float(retrieval_results_df["hadith_hit"].mean())
        if "retrieval_results_df" in globals()
        else None,
    }
    summary_path = EVAL_RUN_DIR / "run_summary.json"
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

    print("RAGAS (mean over samples):")
    for k, v in ragas_mean.items():
        print(f"  {k}: {v:.4f}")
    print(f"Saved -> {per_row_path}, {summary_path}")
    ragas_per_question_df.head()
else:
    print("RAGAS eval disabled in EVAL_CONFIG")

KeyboardInterrupt: 

In [ ]:
# --- compare runs (after trying different CONFIG / re-indexing) ---
import json


def load_eval_summaries(runs_root: Path | None = None) -> pd.DataFrame:
    runs_root = runs_root or EVAL_CONFIG["output_dir"]
    rows = []
    for p in sorted(runs_root.glob("*/run_summary.json")):
        data = json.loads(p.read_text(encoding="utf-8"))
        row = {"run": p.parent.name, **data.get("ragas_mean", {})}
        row["hadith_hit_rate"] = data.get("hadith_hit_rate")
        rows.append(row)
    return pd.DataFrame(rows)


# Example after 2+ trials:
# compare_df = load_eval_summaries()
# compare_df.sort_values("hadith_hit_rate", ascending=False)